# E5 base corpus encoding


In [ ]:
import json
import os
import time

import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer


SOURCE_CORPUS_DIR = "/kaggle/input/datasets/sukiss/corpus-embedings"
E5_MODEL_NAME = "intfloat/multilingual-e5-base"

OUT_DIR = "/kaggle/working/e5_base_proxy_encoding"

BATCH_DOCS = 64
PASSAGE_MAX_LEN = 192
DTYPE_SAVE = np.float16

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def average_pool(last_hidden_states, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_states.size()).float()
    masked = last_hidden_states * mask
    return masked.sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)


def load_model():
    assert DEVICE == "cuda", "This encoding script is configured for GPU/Kaggle CUDA runtime"

    tokenizer = AutoTokenizer.from_pretrained(E5_MODEL_NAME)
    model = AutoModel.from_pretrained(E5_MODEL_NAME)
    model.to(DEVICE)
    model.eval()

    print("Loaded E5 base model:", E5_MODEL_NAME)
    return tokenizer, model


@torch.inference_mode()
def encode_texts(tokenizer, model, texts, max_len, is_query=False):
    prefix = "query: " if is_query else "passage: "
    texts = [prefix + x for x in texts]

    batch = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt",
    )
    batch = {k: v.to(DEVICE) for k, v in batch.items()}

    out = model(**batch)
    emb = average_pool(out.last_hidden_state, batch["attention_mask"])
    emb = F.normalize(emb, p=2, dim=-1)
    return emb.detach().cpu().numpy()


def load_corpus_jsonl(path):
    docids, texts = [], []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            docids.append(str(obj["docid"]))
            texts.append(obj["text"])

    return docids, texts


def encode_corpus(tokenizer, model, name):
    src_path = os.path.join(SOURCE_CORPUS_DIR, f"{name}_corpus.jsonl")
    out_corpus = os.path.join(OUT_DIR, f"{name}_corpus.jsonl")
    out_docids = os.path.join(OUT_DIR, f"{name}_docids.txt")
    out_emb = os.path.join(OUT_DIR, f"{name}_embeddings.npy")

    assert os.path.exists(src_path), src_path

    if os.path.exists(out_emb) and os.path.exists(out_docids) and os.path.exists(out_corpus):
        print(f"[{name}] already encoded, skipping")
        return

    docids, texts = load_corpus_jsonl(src_path)
    if not texts:
        raise ValueError(f"[{name}] corpus is empty")

    print(f"[{name}] docs:", len(docids))

    with open(out_corpus, "w", encoding="utf-8") as f:
        for did, text in zip(docids, texts):
            f.write(json.dumps({"docid": did, "text": text}, ensure_ascii=False) + "\n")

    with open(out_docids, "w", encoding="utf-8") as f:
        for did in docids:
            f.write(did + "\n")

    first = encode_texts(tokenizer, model, texts[:1], PASSAGE_MAX_LEN, is_query=False).astype(DTYPE_SAVE)
    dim = first.shape[1]

    emb = np.memmap(out_emb, dtype=DTYPE_SAVE, mode="w+", shape=(len(texts), dim))
    emb[0:1] = first

    start = time.time()
    for i in tqdm(range(1, len(texts), BATCH_DOCS), desc=f"encoding {name}"):
        j = min(len(texts), i + BATCH_DOCS)
        emb[i:j] = encode_texts(tokenizer, model, texts[i:j], PASSAGE_MAX_LEN, is_query=False).astype(DTYPE_SAVE)

    emb.flush()

    print(f"[{name}] saved:")
    print(" corpus:", out_corpus)
    print(" docids:", out_docids)
    print(" emb:", out_emb, "dim:", dim, "MB:", round(os.path.getsize(out_emb) / 1024 / 1024, 2))
    print(" time_s:", round(time.time() - start, 1))


def main():
    os.makedirs(OUT_DIR, exist_ok=True)

    tokenizer, model = load_model()
    encode_corpus(tokenizer, model, "main")
    encode_corpus(tokenizer, model, "test")

    print("DONE:", sorted(os.listdir(OUT_DIR)))


if __name__ == "__main__":
    main()
